# Export images of the TREX datalake to Google Drive (example: Lake Balaton)
This notebook extracts the Sentinel-2 images processed on the CNES cluster **TREX** (tree of the TREX guide,
`/work/datalake/watcal/Sentinel-2/{L1C,L2A,L2B}`, written by {doc}`grs_datalake_trex_batch_v3`) over an
area given by its **coordinates**, between a **start and an end date**, for one or several **levels**. The
example is **Lake Balaton** (Hungary, 77 km long), which spans **six Sentinel-2 tiles on both sides of the
boundary between the UTM zones 33 and 34** (33TXM, 33TXN, 33TYM, 33TYN, 34TBS, 34TBT): every tile intersecting
the lake is exported, tile by tile, each in its own grid.

| level | source | exported file(s), one per image (tile and date) |
|---|---|---|
| `l1c` | L1C `.SAFE` folders synchronised by `sync_l1c.py` (JPEG2000 bands, digital numbers) | `<name>_<site>.zip`: three GeoTIFFs (bands at 10, 20 and 60 m, `uint16` digital numbers, as in the product) and the metadata `MTD_MSIL1C.xml` / `MTD_TL.xml` (offsets, quantification, angles) |
| `l2a` | GRS L2A products (NetCDF or Zarr) | `<name>_<site>/<name>_<site>.nc` and `_anc.nc` (GRS packing, readable by {class}`~grstbx.driver.L2grs`); option: GeoTIFF of $R_{rs}$ |
| `l2b` | GRSl2bgen L2B products (NetCDF or Zarr) | `<name>_<site>.nc` (`int16` packing of the product) |

The exports are then copied to a **Google Drive** folder with `rclone`. Zarr stores are thousands of small
files, slow to upload and to sync on a Drive: the default format is **NetCDF**; Zarr is available as a
single **zipped store** (`.zarr.zip`, not compressed again, readable without unzipping, see the end).

**Areas over several tiles** (`tile_choice`): with `'all'` (default), every tile intersecting the area is
exported, **tile by tile**, each cut to the area in its own grid (UTM zone of the tile): nothing is
resampled, the exports are lossless, and they can be mosaicked afterwards if needed. With `'best'`, only the
tile covering the largest part of the area is exported for each acquisition (area within one tile).

Steps:

1. **Configuration**: area, dates, levels, tiles, formats, Drive folder.
2. **Tiles** intersecting the area (footprints from the STAC Items of the products, the products themselves,
   or the L1C metadata; cached in `<export_dir>/tile_footprints.json`).
3. **Images** of these tiles in the date range, one per acquisition, tile and level (latest GRS version, or
   `grs_version`).
4. **Export**: each product is opened lazily, cut to the area (only the chunks / JPEG2000 tiles within it are
   read) and written. Exports already done are skipped; acquisitions without data over
   the area (edge of the swath) are left out (status `empty`).
5. **Upload** to Google Drive with `rclone copy` (only new or changed files are sent), with the list of the
   exports (`manifest_<start>_<end>.csv`) and the area (`aoi.geojson`).
6. **Check**: one export of each level reopened.
7. **SLURM job array** for long periods (one task per year).

**On TREX.** The exports run on a compute node (SLURM job of section 7, or an interactive `srun` session):
they only read `/work/datalake` and write to your scratch space. The **compute nodes have no internet
access**: the upload to Google Drive (section 5) runs on a **login node**, through the proxy of CNES, or from
your workstation (rclone over SFTP), see section 5.

The notebook runs in the environment `grs_v3_venv` of the TREX guide (GRS 3.x for the L2A writers, grstbx,
rioxarray); start it after `unset PYTHONPATH; export PYTHONNOUSERSITE=1` (Spack modules).

In [ ]:
import os
import re
import glob
import json
import shutil
import zipfile
import subprocess
import warnings
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd
import xarray as xr
import rioxarray  # noqa: F401, rio accessor
import geopandas as gpd
import shapely
from shapely.geometry import box
from pathlib import Path
from packaging.version import Version, InvalidVersion

import grstbx
from grstbx import batch
from grstbx.subset import _sanitize      # attributes NetCDF cannot store, as strings

opj = os.path.join
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', message='The specified chunks separate the stored chunks')
warnings.filterwarnings('ignore', message='Consolidated metadata is currently not part')

# TREX: Spack modules put system paths in PYTHONPATH (guide, section 2.1)
if os.environ.get('PYTHONPATH'):
    warnings.warn('PYTHONPATH is set: start Jupyter (or the job) after `unset PYTHONPATH; export PYTHONNOUSERSITE=1`')

## 1. Configuration

| Parameter | Description |
|---|---|
| `site` | name of the area, used in the names of the exports and of the Drive folder |
| `lon`, `lat`, `width`, `height` | box centred on a point (degrees, metres); or `bbox` = (lon min, lat min, lon max, lat max); or `aoi_file`, a vector file (GeoJSON, shapefile...) |
| `start_date`, `end_date` | date range, inclusive (env. `EXPORT_START`, `EXPORT_END`) |
| `levels` | among `'l1c'`, `'l2a'`, `'l2b'` |
| `tiles` | tiles to look at (`None`: every tile of the archive intersecting the area) |
| `tile_choice` | `'all'`: every tile intersecting the area, tile by tile; `'best'`: the tile covering the area best, for each acquisition |
| `grs_version` | GRS version of the L2A / L2B products (e.g. `'V3.1.1'`); `None`: the latest of each image |
| `fmt` | `'nc'` (NetCDF, default) or `'zarr.zip'` (zipped Zarr store) for L2A and L2B |
| `l2a_rrs_geotiff` | also write $R_{rs}$ of the L2A products as a float32 GeoTIFF (`water_only`: valid water pixels only) |
| `l2b_variables` | L2B variables to keep (`None`: all) |
| `mask_outside` | pixels of the bounding box outside the polygon set as missing (L2A: GRS conventions) |
| `max_cloud_cover` | images at least this cloudy (0-1, from the STAC Item or the L1C metadata) left out; `None`: all |
| `data_root` | root of the products on TREX: `<data_root>/{L1C,L2A,L2B}/<tile>/YYYY/MM/DD/` (env. `EXPORT_DATA_ROOT`) |
| `export_dir` | folder of the exports, `<export_dir>/<site>/<level>/`, in your scratch space (env. `EXPORT_DIR`) |
| `drive_remote`, `drive_folder` | `rclone` remote of the Google Drive and folder in it: the exports go to `<drive_remote><drive_folder>/<site>/` |
| `upload`, `dry_run` | copy to the Drive; `dry_run=True` only lists what would be copied |

In [ ]:
site = 'balaton'
lon, lat, width, height = None, None, None, None    # box centred on a point (deg, m)
bbox = (17.22, 46.69, 18.21, 47.11)                  # Lake Balaton
aoi_file = None                  # e.g. a polygon of the lake shore (GeoJSON) to mask the land

start_date = os.environ.get('EXPORT_START', '2024-06-01')
end_date = os.environ.get('EXPORT_END', '2024-09-30')
levels = ['l2a', 'l2b']          # 'l1c' too if the L1C images were synchronised (sync_l1c.py)
tiles = ['33TXM', '33TXN', '33TYM', '33TYN', '34TBS', '34TBT']   # None: all the tiles intersecting the area
tile_choice = 'all'              # 'all' (every tile intersecting the area) or 'best'
grs_version = None               # e.g. 'V3.1.1'; None: latest version of each image

fmt = 'nc'                       # 'nc' or 'zarr.zip' (L2A and L2B)
l2a_rrs_geotiff = False          # also Rrs as GeoTIFF
water_only = True                # for the Rrs GeoTIFF
l2b_variables = None             # e.g. ['Chla_OWTblend_Ta2025', 'SPM_OWTblend_Ta2025', 'TURB_dogliotti', 'owt_index_Ta2025']
mask_outside = True
max_cloud_cover = None           # e.g. 0.8
overwrite = False

data_root = os.environ.get('EXPORT_DATA_ROOT', '/work/datalake/watcal/Sentinel-2')
LEVEL_ROOT = {level: opj(data_root, level.upper()) for level in ('l1c', 'l2a', 'l2b')}
user = os.environ.get('USER', 'user')
export_dir = os.environ.get('EXPORT_DIR', f'/work/scratch/data/{user}/exports')

drive_remote = 'mag_watqual:'    # rclone remote of the Google Drive (section 5)
drive_folder = 'grstbx_exports'
upload = os.environ.get('EXPORT_UPLOAD', '1') == '1'    # on a login node only (no internet on the compute nodes)
dry_run = False
site_dir = opj(export_dir, site)
period = f'{start_date}_{end_date}'
os.makedirs(site_dir, exist_ok=True)
print(f'{site}: {start_date} to {end_date}, levels {levels}, tile_choice {tile_choice!r} -> {site_dir}')

The area, as a GeoDataFrame in WGS84 (`aoi`), and its outline:

In [ ]:
def write_atomic(path, write):
    """Write a file under a temporary name, then rename it (several jobs may write the same file)."""
    tmp = f'{path}.{os.getpid()}.part'
    write(tmp)
    os.replace(tmp, path)


if aoi_file:
    aoi = gpd.read_file(aoi_file).to_crs(4326)
elif bbox:
    aoi = gpd.GeoDataFrame(geometry=[box(*bbox)], crs=4326)
else:
    aoi = grstbx.aoi_from_box(lon, lat, width=width, height=height)
local_crs = aoi.estimate_utm_crs()          # metric CRS for the areas
aoi_local = shapely.unary_union(aoi.to_crs(local_crs).geometry)
write_atomic(opj(site_dir, 'aoi.geojson'), lambda p: aoi.to_file(p, driver='GeoJSON'))
print(f'{site}: bounds {np.round(aoi.total_bounds, 4)}, {aoi_local.area / 1e6:.0f} km2')
aoi.plot(edgecolor='k', facecolor='none');

## 2. Tiles intersecting the area

The footprint of a tile (UTM projection and bounding box of its 110 × 110 km grid) is read from one of its
products: the STAC Item next to an L2B or L2A product (`proj:code`, `proj:bbox`), else the product itself
(NetCDF or Zarr), else the `MTD_TL.xml` file of an L1C image. The footprints are cached in
`<export_dir>/tile_footprints.json`, so the archive is looked at only once per tile. `cover` is the fraction
of the area within the tile.

In [ ]:
FOOTPRINT_CACHE = opj(export_dir, 'tile_footprints.json')


def first_product(root, tile):
    """Latest product folder of a tile, <root>/<tile>/YYYY/MM/DD/<product>, without listing the whole tree."""
    path = opj(root, tile)
    for depth in range(4):
        if not os.path.isdir(path):
            return None
        entries = sorted(e for e in os.listdir(path) if not e.startswith('.'))
        if not entries:
            return None
        path = opj(path, entries[-1])      # latest year, month, day, product
    return path


def open_product(path):
    """L2A or L2B product, NetCDF file or Zarr store (full-resolution level of a pyramid), lazily."""
    if path.endswith('.zarr'):
        groups = [d for d in os.listdir(path) if d.isdigit()]
        return xr.open_zarr(path, group='0' if groups else None, decode_coords='all')
    return xr.open_dataset(path, decode_coords='all')


def product_file(folder):
    """Data of a product folder: <name>.zarr or <name>.nc (None if absent); the .SAFE folder for L1C."""
    name = os.path.basename(folder)
    if name.endswith('.SAFE'):
        return folder if os.path.exists(opj(folder, 'MTD_MSIL1C.xml')) else None
    for ext in ('.zarr', '.nc'):
        path = opj(folder, name + ext)
        if os.path.exists(path):
            return path
    return None


def l1c_geocoding(safe):
    """(EPSG, UTM bounding box) of an L1C tile, from its MTD_TL.xml."""
    mtd = glob.glob(opj(safe, 'GRANULE', '*', 'MTD_TL.xml'))[0]
    root = ET.parse(mtd).getroot()
    epsg = int(root.find('.//HORIZONTAL_CS_CODE').text.split(':')[1])
    geo = root.find(".//Geoposition[@resolution='10']")
    ulx, uly = float(geo.find('ULX').text), float(geo.find('ULY').text)
    return epsg, [ulx, uly - 109800, ulx + 109800, uly]


def product_footprint(folder):
    """(EPSG, UTM bounding box) of a product folder: STAC Item, product, or L1C metadata; None if unknown."""
    name = os.path.basename(folder)
    item = opj(folder, name + '.json')
    if os.path.exists(item):
        props = json.load(open(item))['properties']
        return int(props['proj:code'].split(':')[1]), list(props['proj:bbox'])
    path = product_file(folder)
    if path is None:
        return None
    if path.endswith('.SAFE'):
        return l1c_geocoding(path)
    ds = open_product(path)
    return ds.rio.crs.to_epsg(), list(ds.rio.bounds())


def tile_footprint(tile):
    for level in ('l2b', 'l2a', 'l1c'):
        product = first_product(LEVEL_ROOT[level], tile)
        if product:
            try:
                footprint = product_footprint(product)
            except Exception:
                footprint = None
            if footprint:
                return footprint
    return None


cache = json.load(open(FOOTPRINT_CACHE)) if os.path.exists(FOOTPRINT_CACHE) else {}
if tiles:
    candidates = sorted(tiles)
else:
    candidates = sorted({t for level in levels if os.path.isdir(LEVEL_ROOT[level])
                         for t in os.listdir(LEVEL_ROOT[level]) if re.fullmatch(r'\d{2}[A-Z]{3}', t)})
rows = []
for tile in candidates:
    if cache.get(tile) is None:            # unknown footprints are not cached: products may come later
        footprint = tile_footprint(tile)
        if footprint is None:
            continue
        cache[tile] = footprint
    epsg, bounds = cache[tile]
    footprint = gpd.GeoSeries([box(*bounds)], crs=epsg).to_crs(local_crs).iloc[0]
    cover = footprint.intersection(aoi_local).area / aoi_local.area
    if cover > 0:
        rows.append(dict(tile=tile, epsg=epsg, ulx=bounds[0], uly=bounds[3], cover=round(cover, 3),
                         footprint=footprint))
write_atomic(FOOTPRINT_CACHE, lambda p: Path(p).write_text(json.dumps(cache)))
tile_table = pd.DataFrame(rows, columns=['tile', 'epsg', 'ulx', 'uly', 'cover', 'footprint'])
tile_table = tile_table.sort_values('cover', ascending=False).reset_index(drop=True)
if tile_table.empty:
    raise ValueError(f'no tile of the archive intersects the area of {site} (tiles looked at: {candidates})')
missing = sorted(set(tiles or []) - set(tile_table.tile))
if missing:
    print(f'tiles without products or outside the area: {missing}')
tile_table.drop(columns='footprint')

## 3. Images in the date range

The day folders `<tile>/YYYY/MM/DD/` of each level are listed between the two dates. For L2A and L2B, the
product must be complete ({func}`~grstbx.batch.l2a_complete`, STAC Item for L2B) and, when an image was
processed with several GRS versions, `grs_version` or else the latest one is kept. The cloud cover comes from
the STAC Item (L2A / L2B) or `MTD_MSIL1C.xml` (L1C).

In [ ]:
VERSION = re.compile(r'_V(\d+(?:\.\d+)*\w*)$')


def version_of(name):
    match = VERSION.search(name)
    try:
        return Version(match.group(1)) if match else Version('0')
    except InvalidVersion:
        return Version('0')


def cloud_cover(level, path):
    """Cloud cover 0-1 from the STAC Item (L2A / L2B, in percent) or MTD_MSIL1C.xml (L1C), NaN if unknown."""
    try:
        if level == 'l1c':
            return batch.cloud_cover(path)
        props = json.load(open(batch.item_path(path)))['properties']
        return float(props.get('eo:cloud_cover', np.nan)) / 100
    except Exception:
        return np.nan


def list_images(level, tile, start, end):
    rows = []
    for day in pd.date_range(start, end, freq='D'):
        folder = opj(LEVEL_ROOT[level], tile, f'{day:%Y/%m/%d}')
        if not os.path.isdir(folder):
            continue
        for entry in sorted(os.listdir(folder)):
            path = product_file(opj(folder, entry))
            if path is None:
                continue
            name = batch.image_name(path)
            if level != 'l1c' and grs_version and not name.endswith('_' + grs_version):
                continue
            if level == 'l2a' and not batch.l2a_complete(path):
                continue
            if level == 'l2b' and not batch.l2b_complete(path):
                continue
            parts = name.split('_')       # S2A, MSIL1C / MSIL2AGRS / MSIL2B, sensing time, baseline, orbit, ...
            rows.append(dict(level=level, tile=tile, date=day, name=name, path=path, version=version_of(name),
                             datatake=f'{parts[0]}_{parts[2]}'))
    return rows


rows = [r for level in levels for tile in tile_table.tile if os.path.isdir(LEVEL_ROOT[level])
        for r in list_images(level, tile, start_date, end_date)]
images = pd.DataFrame(rows, columns=['level', 'tile', 'date', 'name', 'path', 'version', 'datatake'])
if not images.empty:
    images = images.merge(tile_table[['tile', 'cover']], on='tile')
    # one product per image: grs_version, or the latest version
    images = images.sort_values(['version', 'name']).drop_duplicates(['level', 'tile', 'datatake'], keep='last')
    if tile_choice == 'best':
        images = images.sort_values('cover').drop_duplicates(['level', 'datatake'], keep='last')
    images['cloud_cover'] = [cloud_cover(l, p) for l, p in zip(images.level, images.path)]
    if max_cloud_cover is not None:
        images = images[~(images.cloud_cover >= max_cloud_cover)]
    images = images.sort_values(['level', 'date', 'cover'], ascending=[True, True, False]).reset_index(drop=True)
if images.empty:
    print('no image')
else:
    print(images.groupby(['level', 'tile']).size().unstack(fill_value=0).to_string())
images.drop(columns=['version'])

## 4. Export

- **L1C**: each band (JPEG2000, `uint16` digital numbers) is cut to the bounding box of the area in the UTM
  projection of the tile, at its native resolution, and the bands are gathered by resolution into three
  GeoTIFFs (`B02 B03 B04 B08` at 10 m, `B05 B06 B07 B8A B11 B12` at 20 m, `B01 B09 B10` at 60 m; deflate
  compression, band names in the descriptions). They are zipped with `MTD_MSIL1C.xml` and `MTD_TL.xml`,
  which give the quantification value, the radiometric offsets (baseline 04.00 and later) and the angles
  needed to compute the TOA reflectance.
- **L2A**: {func}`grstbx.crop_l2a` (lazy crop, pixels outside the polygon flagged with `mask_outside`) and
  {func}`grstbx.export_l2a`: GRS layout and packing, lossless, with the quick looks of GRS (PNG) next to the
  NetCDF files.
- **L2B**: full-resolution level of the pyramid, cut to the bounding box, written with the `int16` packing
  (`scale_factor`, `add_offset`) of the product, compressed.

With `fmt='zarr.zip'`, the L2A and L2B products are written as Zarr stores, then zipped without compression
(the chunks are already compressed) into one `<name>_<site>.zarr.zip` file. Every export is written under a
temporary name and renamed once complete, so an interrupted run leaves nothing behind.

In [ ]:
L1C_GROUPS = {10: ['B02', 'B03', 'B04', 'B08'], 20: ['B05', 'B06', 'B07', 'B8A', 'B11', 'B12'],
              60: ['B01', 'B09', 'B10']}
ENCODING_KEYS = ('dtype', 'scale_factor', 'add_offset', '_FillValue', 'units', 'calendar')
NODATA_BIT = 1          # bit 0 of the GRS flags


def bounds_in(crs, geometry=None):
    """Bounds of the area (or of a GeoSeries) in a CRS."""
    return (aoi if geometry is None else geometry).to_crs(crs).total_bounds


def zip_folder(folder, archive):
    """Zip a Zarr store without compression (its chunks are compressed), its content at the root of the
    archive, as zarr.storage.ZipStore expects."""
    tmp = archive + '.part'
    with zipfile.ZipFile(tmp, 'w', zipfile.ZIP_STORED, allowZip64=True) as z:
        for root, _, names in os.walk(folder):
            for n in names:
                path = opj(root, n)
                z.write(path, os.path.relpath(path, folder))
    os.replace(tmp, archive)
    shutil.rmtree(folder)
    return archive


def l1c_bands(safe):
    granule = glob.glob(opj(safe, 'GRANULE', '*'))[0]
    jp2 = {re.search(r'_(B\w\w)\.jp2$', f).group(1): f for f in glob.glob(opj(granule, 'IMG_DATA', '*_B??.jp2'))}
    return granule, jp2


def l1c_stack(jp2, bands, clip):
    """Bands of one resolution cut to `clip(crs) -> bounds`, stacked on a 'band' dimension."""
    arrays = []
    for b in bands:
        da = rioxarray.open_rasterio(jp2[b], chunks=True)
        arrays.append(da.rio.clip_box(*clip(da.rio.crs)).squeeze('band', drop=True))
    return xr.concat(arrays, dim='band').assign_coords(band=np.arange(1, len(bands) + 1)).load()


def write_l1c_zip(out, name, stacks, metadata):
    """GeoTIFFs of the stacks {res: DataArray} and metadata files {arcname: path} zipped into `out`."""
    work = out + '.dir'
    os.makedirs(work, exist_ok=True)
    try:
        files = {}
        for res, stack in stacks.items():
            stack.attrs = {'long_name': tuple(stack.attrs['bands']), 'units': 'digital number'}
            tif = opj(work, f'{name}_{res}m.tif')
            stack.rio.write_nodata(0).rio.to_raster(tif, compress='deflate', predictor=2, tiled=True)
            files[os.path.basename(tif)] = tif
        files.update(metadata)
        with zipfile.ZipFile(out + '.part', 'w', zipfile.ZIP_DEFLATED) as z:
            for arcname, path in files.items():
                z.write(path, arcname)
        os.replace(out + '.part', out)
    finally:
        shutil.rmtree(work, ignore_errors=True)


def l2b_encoding(ds, packed=True):
    """Encoding of an L2B dataset: packing of the product (`packed`) or float32, compressed for NetCDF."""
    encoding = {}
    for v in ds.variables:
        spatial = {'x', 'y'} <= set(ds[v].dims)
        if packed:
            enc = {k: ds[v].encoding[k] for k in ENCODING_KEYS if k in ds[v].encoding}
        else:
            enc = {'dtype': 'float32', '_FillValue': np.float32(np.nan)} if spatial and ds[v].dtype.kind == 'f' else {}
        if spatial and fmt != 'zarr.zip':
            enc.update(zlib=True, complevel=4)
        encoding[v] = enc
    return encoding


def write_l2b(ds, out, name, out_dir, packed=True):
    encoding = l2b_encoding(ds, packed)
    ds = _sanitize(ds)
    for v in ds.variables:
        ds[v].encoding = {}
    if fmt == 'zarr.zip':
        store = opj(out_dir, f'.{name}.zarr')
        shutil.rmtree(store, ignore_errors=True)
        ds.chunk({'x': 512, 'y': 512}).to_zarr(store, encoding=encoding, consolidated=True)
        zip_folder(store, out)
    else:
        ds.to_netcdf(out + '.part', encoding=encoding)
        os.replace(out + '.part', out)


def write_l2a(ds, ancillary, out, name, out_dir):
    """GRS writers, in a hidden folder moved in place once complete."""
    work = opj(out_dir, f'.{name}.part')
    shutil.rmtree(work, ignore_errors=True)
    try:
        if fmt == 'zarr.zip':
            store = grstbx.export_l2a(ds, opj(work, name + '.zarr'), ancillary=ancillary)
            zip_folder(store, out)
        else:
            grstbx.export_l2a(ds, opj(work, name + '.nc'), ancillary=ancillary)
            shutil.rmtree(os.path.dirname(out), ignore_errors=True)
            os.replace(work, os.path.dirname(out))
        if l2a_rrs_geotiff:
            grstbx.export_rrs_geotiff(ds, opj(out_dir, f'{name}_Rrs.tif'), water_only=water_only)
    finally:
        shutil.rmtree(work, ignore_errors=True)


def out_path(level, name, out_dir):
    if level == 'l1c':
        return opj(out_dir, name + '.zip')
    if fmt == 'zarr.zip':
        return opj(out_dir, name + '.zarr.zip')
    return opj(out_dir, name, name + '.nc') if level == 'l2a' else opj(out_dir, name + '.nc')


# -- export of one image ---------------------------------------------------------------------------------------------

def export_single(level, row, out_dir):
    name = f"{row['name']}_{site}"
    out = out_path(level, name, out_dir)
    if os.path.exists(out) and not overwrite:
        return out, 'exists'
    if level == 'l1c':
        granule, jp2 = l1c_bands(row['path'])
        stacks = {}
        for res, bands in L1C_GROUPS.items():
            bands = [b for b in bands if b in jp2]
            stacks[res] = l1c_stack(jp2, bands, bounds_in).assign_attrs(bands=bands)
        if not any(bool((s > 0).any()) for s in stacks.values()):
            return None, 'empty'
        write_l1c_zip(out, name, stacks, {'MTD_MSIL1C.xml': opj(row['path'], 'MTD_MSIL1C.xml'),
                                          'MTD_TL.xml': opj(granule, 'MTD_TL.xml')})
    elif level == 'l2a':
        product, ancillary = grstbx.open_l2a(row['path'])
        subset, subset_anc = grstbx.crop_l2a(product, aoi, ancillary, mask_outside=mask_outside)
        subset = subset.load()
        if not bool((subset['mask'] == 0).any()):
            return None, 'empty'
        write_l2a(subset, subset_anc, out, name, out_dir)
    else:
        ds = open_product(row['path'])
        if l2b_variables:
            ds = ds[[v for v in l2b_variables if v in ds]]
        subset = ds.rio.clip_box(*bounds_in(ds.rio.crs)).load()
        maps = [v for v in subset.data_vars if {'x', 'y'} <= set(subset[v].dims) and subset[v].dtype.kind == 'f']
        if not any(bool(subset[v].notnull().any()) for v in maps):
            return None, 'empty'
        write_l2b(subset, out, name, out_dir, packed=True)
    return out, 'done'


# -- run ------------------------------------------------------------------------------------------------

def export_size(out, level):
    if not out or not os.path.exists(out):
        return 0.
    if level == 'l2a' and fmt == 'nc':
        folder = os.path.dirname(out)
        return sum(os.path.getsize(opj(folder, f)) for f in os.listdir(folder)) / 2**20
    return os.path.getsize(out) / 2**20


records = []
for i, row in images.iterrows():
    level = row['level']
    out_dir = opj(site_dir, level)
    os.makedirs(out_dir, exist_ok=True)
    try:
        out, status = export_single(level, row, out_dir)
        error = ''
    except Exception as e:
        out, status, error = None, 'failed', f'{type(e).__name__}: {e}'
    records.append(dict(level=level, date=row['date'].date(), tile=row['tile'], name=row['name'],
                        cloud_cover=row['cloud_cover'], status=status,
                        export=os.path.relpath(out, site_dir) if out else '',
                        size_mb=round(export_size(out, level), 1), source=row['path'], error=error))
    print(f"[{i + 1}/{len(images)}] {level} {row['date']:%Y-%m-%d} {row['tile']} {status:7s} {row['name']} {error}",
          flush=True)

manifest = pd.DataFrame(records, columns=['level', 'date', 'tile', 'name', 'cloud_cover', 'status', 'export',
                                          'size_mb', 'source', 'error'])
write_atomic(opj(site_dir, f'manifest_{period}.csv'), lambda p: manifest.to_csv(p, index=False))
manifest.groupby(['level', 'tile', 'status']).agg(images=('name', 'size'), size_mb=('size_mb', 'sum'))

## 5. Upload to Google Drive

The compute nodes of TREX have no internet access: the upload runs on a **login node** (`trexvisu06/07`),
whose outbound HTTPS goes through the proxy of CNES (`http://proxy.cnes.fr:8050`), which intercepts SSL with
its own certificate. `rclone copy` sends `<export_dir>/<site>` to `<drive_remote><drive_folder>/<site>/`
(only the new or changed files).

**Set up once** (on a login node):

1. `rclone`: `module avail rclone`, or the static binary in `~/bin` (download it on your workstation
   from https://rclone.org/downloads/ and copy it with `scp`).
2. The Google Drive remote: OAuth needs a web browser, which TREX has not. On your workstation, where the
   remote already works, copy its section of `~/.config/rclone/rclone.conf` (`rclone config show
   mag_watqual`) into `~/.config/rclone/rclone.conf` on TREX (readable by you only: `chmod 600`); or run
   `rclone config` on TREX and answer *No* to *Use web browser*, then `rclone authorize "drive"` on your
   workstation.
3. The certificate of the proxy: `ca_cert` below, the CA bundle of the system (it holds the CNES authority on
   TREX). If it is refused (`x509: certificate signed by unknown authority`), get the certificate of the CNES
   authority from the HPC support; `insecure=True` (`--no-check-certificate`) works but does not check
   the server.

**Without the proxy**: run rclone on your workstation, reading TREX over SFTP (a remote `trex` of type `sftp`
with your SSH key), e.g. `rclone copy trex:/work/scratch/data/<user>/exports/balaton
mag_watqual:grstbx_exports/balaton`.

In [ ]:
proxy = 'http://proxy.cnes.fr:8050'
ca_cert = next((c for c in ('/etc/pki/ca-trust/extracted/pem/tls-ca-bundle.pem', '/etc/pki/tls/certs/ca-bundle.crt')
                if os.path.exists(c)), None)
insecure = False                 # --no-check-certificate if the certificate of the proxy is refused
rclone = shutil.which('rclone') or os.path.expanduser('~/bin/rclone')

destination = f'{drive_remote}{drive_folder}/{site}'
if not upload:
    print('upload switched off (EXPORT_UPLOAD=0)')
elif os.environ.get('SLURM_JOB_ID'):
    print(f'compute node (no internet access): upload from a login node, this cell or\n  {rclone} copy {site_dir} {destination}')
elif not os.path.exists(rclone):
    print('rclone not found: see the set-up above')
else:
    env = os.environ | {'HTTPS_PROXY': proxy, 'HTTP_PROXY': proxy, 'https_proxy': proxy, 'http_proxy': proxy}
    cmd = [rclone, 'copy', site_dir, destination, '--exclude', '.*/**', '--exclude', '*.part',
           '--exclude', 'tile_footprints.json', '--transfers', '4', '--drive-chunk-size', '64M',
           '--stats-one-line', '-v']
    cmd += ['--no-check-certificate'] if insecure else (['--ca-cert', ca_cert] if ca_cert else [])
    if dry_run:
        cmd.append('--dry-run')
    print(' '.join(cmd))
    result = subprocess.run(cmd, capture_output=True, text=True, env=env)
    print('\n'.join(line for line in result.stderr.splitlines() if 'client_id' not in line)[-3000:])
    if result.returncode:
        raise RuntimeError(f'rclone failed (exit {result.returncode}), see the set-up above')

## 6. Check: reopen one export of each level

- L1C: the GeoTIFFs are read from the zip archive with GDAL (`/vsizip/`), no need to unzip;
- L2A: the main NetCDF file (or the zipped Zarr store);
- L2B: `xarray.open_dataset` (the `int16` values are decoded with their `scale_factor` / `add_offset`).

A zipped Zarr store opens without unzipping: `xr.open_zarr(zarr.storage.ZipStore(path, mode='r'))`.
Skipped in the batch tasks (`EXPORT_BATCH_JOB=1`, set by the SLURM script).

In [ ]:
import zarr
import matplotlib.pyplot as plt


def open_export(path, group=None):
    if path.endswith('.zip'):
        return xr.open_zarr(zarr.storage.ZipStore(path, mode='r'), group=group)
    return xr.open_dataset(path)


done = manifest[manifest.status.isin(['done', 'exists'])].drop_duplicates('level')
if os.environ.get('EXPORT_BATCH_JOB') or done.empty:
    print('no check' + ('' if os.environ.get('EXPORT_BATCH_JOB') else ': nothing exported'))
else:
    fig, axs = plt.subplots(1, len(done), figsize=(6 * len(done), 5), squeeze=False)
    for ax, (_, row) in zip(axs[0], done.iterrows()):
        path = opj(site_dir, row['export'])
        if row['level'] == 'l1c':
            name = os.path.basename(path).removesuffix('.zip')
            da = rioxarray.open_rasterio(f'/vsizip/{path}/{name}_10m.tif')
            rgb = da.sel(band=[3, 2, 1]).astype('float32')
            (rgb / np.nanpercentile(rgb.where(rgb > 0), 98)).clip(0, 1).plot.imshow(ax=ax)
            ax.set_title(f"L1C {row['date']} (B04 B03 B02)")
        elif row['level'] == 'l2a':
            ds = open_export(path, group='0' if path.endswith('.zip') else None)
            ds.Rrs.sel(wl=[665, 560, 490], method='nearest').where(ds['mask'] == 0).plot.imshow(
                rgb='wl', robust=True, ax=ax)
            ax.set_title(f"L2A {row['date']}: Rrs, valid water pixels")
        else:
            ds = open_export(path)
            var = next((v for v in ('Chla_OWTblend_Ta2025', 'TURB_dogliotti', 'SPM_nechad') if v in ds),
                       list(ds.data_vars)[0])
            ds[var].squeeze().plot.imshow(ax=ax, robust=True, cmap='Spectral_r', cbar_kwargs={'shrink': 0.7})
            ax.set_title(f"L2B {row['date']}: {var}")
        ax.set_aspect('equal')
        ax.set_axis_off()
    plt.tight_layout()

## 7. Long periods: SLURM job array on TREX

For years of images, the exports run as a **SLURM job array**, one task per year (`EXPORT_START`,
`EXPORT_END`), with this notebook run by IPython on the compute nodes (`EXPORT_UPLOAD=0`: no internet there).
The exports already done are skipped, so the array can be submitted again; each task writes its own
`manifest_<start>_<end>.csv`. Then upload from a login node (section 5).

Run without `sbatch`, the script prints the `sbatch` command (dry run). Set the project account (`-A`).

In [ ]:
notebook = os.path.abspath('grstbx_trex_export_gdrive.ipynb')    # this notebook
account = '<PROJECT_ACCOUNT>'
script = f"""#!/bin/bash
#SBATCH --job-name=export_{site}
#SBATCH --output=logs/export_%A_%a.out
#SBATCH --error=logs/export_%A_%a.err
#SBATCH -N 1
#SBATCH -n 1
#SBATCH -c 4
#SBATCH --mem=32G
#SBATCH --time=04:00:00
#SBATCH -A {account}
#SBATCH -p cpu2022
#SBATCH --export=none
# Export of {site} ({' '.join(levels)}), one task per year.
# Usage: run_export_{site}.slurm <first year> <last year>   (dry run: sbatch command)

set -eo pipefail
export LC_ALL=C
cd "${{SLURM_SUBMIT_DIR:-$PWD}}"
mkdir -p logs
FIRST=$1; LAST=$2
[[ -z "$LAST" ]] && {{ echo "Usage: $0 <first year> <last year>"; exit 1; }}
N=$(( LAST - FIRST + 1 ))
if [[ -z "${{SLURM_ARRAY_TASK_ID:-}}" ]]; then
    echo "  sbatch --array=0-$((N - 1))%4 $0 $*"
    exit 0
fi
YEAR=$(( FIRST + SLURM_ARRAY_TASK_ID ))
echo "[$(date +'%Y-%m-%d %H:%M:%S')] Job $SLURM_ARRAY_JOB_ID task $SLURM_ARRAY_TASK_ID: {site} $YEAR on $(hostname)"

source ~/.bashrc 2>/dev/null || true
module purge
module load python/3.11.10 gcc/13.2.0
export LD_PRELOAD=$(gcc -print-file-name=libstdc++.so.6)
source ~/envs/grs_v3_venv/bin/activate
unset PYTHONPATH
export PYTHONNOUSERSITE=1
export EXPORT_START=$YEAR-01-01 EXPORT_END=$YEAR-12-31 EXPORT_UPLOAD=0
export EXPORT_DIR={export_dir} EXPORT_DATA_ROOT={data_root}
export OMP_NUM_THREADS=4 GDAL_NUM_THREADS=4 GDAL_CACHEMAX=1024 GDAL_DISABLE_READDIR_ON_OPEN=EMPTY_DIR
export MALLOC_ARENA_MAX=2 MPLBACKEND=Agg
export EXPORT_BATCH_JOB=1    # batch task (JupyterHub on TREX also runs in a SLURM job)

if ipython --no-banner --colors=NoColor {notebook}; then    # stops at the first error, exit code 1
    echo "[$(date +'%Y-%m-%d %H:%M:%S')] COMPLETED"
else
    echo "[$(date +'%Y-%m-%d %H:%M:%S')] FAILED"; exit 1
fi
"""
if os.environ.get('EXPORT_BATCH_JOB'):
    print('in a batch task: script not written')
else:
    script_file = opj(os.path.dirname(notebook), f'run_export_{site}.slurm')
    with open(script_file, 'w') as f:
        f.write(script)
    os.chmod(script_file, 0o755)
    print(f'written {script_file}; e.g. for 2017-2025:')
    print(subprocess.run([script_file, '2017', '2025'], capture_output=True, text=True).stdout)